In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.optim as optim

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# =========================================
# CLEAR GPU MEMORY
# =========================================
torch.cuda.empty_cache()
import gc
gc.collect()

# =========================================
# DEVICE
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =========================================
# TRANSFORMS
# =========================================

# Train transforms (augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Validation / Test transforms (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# =========================================
# LOAD FULL DATASET
# =========================================
root_path = "/kaggle/input/dl-project"

full_dataset = datasets.ImageFolder(root_path)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

# =========================================
# TRAIN/VAL/TEST SPLIT
# =========================================
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_set, val_set, test_set = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train size: {train_size}")
print(f"Val size:   {val_size}")
print(f"Test size:  {test_size}")

# =========================================
# ASSIGN TRANSFORMS CORRECTLY
# =========================================
# NOTE: Each subset has a reference to full_dataset.
# So we must assign transform on the underlying dataset for each split.

train_set.dataset.transform = train_transform
val_set.dataset.transform  = test_transform
test_set.dataset.transform = test_transform

# =========================================
# DATALOADERS
# =========================================
train_loader = DataLoader(
    train_set, batch_size=4, shuffle=True,
    num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_set, batch_size=4,
    num_workers=2, pin_memory=True
)

test_loader = DataLoader(
    test_set, batch_size=4,
    num_workers=2, pin_memory=True
)

print("Dataloaders are ready.")
print(f"Number of classes: {num_classes}")


Using device: cuda
Classes: ['advanced_glaucoma', 'early_glaucoma', 'normal_control']
Train size: 1080
Val size:   231
Test size:  233
Dataloaders are ready.
Number of classes: 3


In [3]:

# Model - Simple DenseNet121
print("Loading model...")
model = models.densenet121(weights='IMAGENET1K_V1')
model.classifier = nn.Linear(model.classifier.in_features, 3)
model = model.to(device)

print(f"Model loaded. GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Loading model...


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 170MB/s] 


Model loaded. GPU memory used: 0.03 GB


In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch

# ===========================
# Training with TQDM
# ===========================

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for imgs, labels in progress_bar:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    
    return total_loss / len(loader)


# ===========================
# Evaluation with TQDM
# ===========================

def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss = 0
    correct = total = 0
    
    progress_bar = tqdm(loader, desc="Validating", leave=False)
    
    with torch.no_grad():
        for imgs, labels in progress_bar:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            
            if criterion:
                batch_loss = criterion(outputs, labels).item()
                total_loss += batch_loss
                progress_bar.set_postfix(loss=batch_loss)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total
    loss = total_loss / len(loader) if criterion else None
    return accuracy, loss


# ===========================
# Training Loop (with Best Model Tracking)
# ===========================

num_epochs = 30
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_losses = []
val_losses = []
val_accuracies = []

best_val_acc = 0
best_model_state = None

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_acc, val_loss = evaluate(model, val_loader, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict()
        torch.save(best_model_state, "best_model.pth")
        print(f"🔥 Best model updated! Saved as best_model.pth (Val Acc: {val_acc:.4f})")
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss  : {val_loss:.4f}")
    print(f"  Val Acc   : {val_acc:.4f}")
    print("------------------------------------")

# ===========================
# Load Best Model Back Into Memory
# ===========================
model.load_state_dict(best_model_state)
print(f"\n🎉 Loaded Best Model (Val Acc = {best_val_acc:.4f})")


# ===========================
# Plot Loss Curves
# ===========================

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.show()

# ===========================
# Plot Accuracy Curve
# ===========================

plt.figure(figsize=(8,5))
plt.plot(val_accuracies, label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Over Epochs")
plt.legend()
plt.grid()
plt.show()



===== Epoch 1/30 =====


🔥 Best model updated! Saved as best_model.pth (Val Acc: 0.8009)
Epoch 1/30
  Train Loss: 0.7710
  Val Loss  : 0.4892
  Val Acc   : 0.8009
------------------------------------

===== Epoch 2/30 =====


Epoch 2/30
  Train Loss: 0.5511
  Val Loss  : 0.5029
  Val Acc   : 0.7835
------------------------------------

===== Epoch 3/30 =====


🔥 Best model updated! Saved as best_model.pth (Val Acc: 0.8052)
Epoch 3/30
  Train Loss: 0.4755
  Val Loss  : 0.4445
  Val Acc   : 0.8052
------------------------------------

===== Epoch 4/30 =====


🔥 Best model updated! Saved as best_model.pth (Val Acc: 0.8398)
Epoch 4/30
  Train Loss: 0.3458
  Val Loss  : 0.4420
  Val Acc   : 0.8398
------------------------------------

===== Epoch 5/30 =====


Epoch 5/30
  Train Loss: 0.2757
  Val Loss  : 0.7988
  Val Acc   : 0.7100
------------------------------------

===== Epoch 6/30 =====


... (17 outputs left out)

Epoch 23/30
  Train Loss: 0.0487
  Val Loss  : 0.5167
  Val Acc   : 0.8355
------------------------------------

===== Epoch 24/30 =====


Epoch 24/30
  Train Loss: 0.0926
  Val Loss  : 0.6886
  Val Acc   : 0.8225
------------------------------------

===== Epoch 25/30 =====


Epoch 25/30
  Train Loss: 0.0970
  Val Loss  : 0.6601
  Val Acc   : 0.8052
------------------------------------

===== Epoch 26/30 =====


Validating:   2%|▏         | 1/58 [00:00<00:06,  8.86it/s, loss=0.246]    

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid()
plt.show()


In [ ]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    cohen_kappa_score,
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

# ======================================================
# COLLECT PREDICTIONS + LABELS + PROBABILITIES
# ======================================================

model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)

        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# ======================================================
# METRICS: ACCURACY, PRECISION, RECALL, F1, KAPPA
# ======================================================

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)
kappa = cohen_kappa_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("\n=== Test Metrics ===")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
print("Kappa    :", kappa)
print("\nConfusion Matrix:\n", cm)

# ======================================================
# MULTICLASS ROC + AUC
# ======================================================

num_classes = len(np.unique(all_labels))

# Binarize labels
y_true_bin = label_binarize(all_labels, classes=list(range(num_classes)))

fpr = {}
tpr = {}
roc_auc = {}

for c in range(num_classes):
    fpr[c], tpr[c], _ = roc_curve(y_true_bin[:, c], all_probs[:, c])
    roc_auc[c] = auc(fpr[c], tpr[c])

# ======================================================
# PLOT ROC CURVE
# ======================================================

plt.figure(figsize=(8, 6))

for c in range(num_classes):
    plt.plot(fpr[c], tpr[c], label=f"Class {c} (AUC = {roc_auc[c]:.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curve")
plt.legend()
plt.grid()
plt.show()

# ======================================================
# SAVE METRICS TO CSV
# ======================================================

metrics_dict = {
    "Accuracy": [accuracy],
    "Precision_weighted": [precision],
    "Recall_weighted": [recall],
    "F1_weighted": [f1],
    "Kappa": [kappa],
}

# Add AUC scores for each class
for c in range(num_classes):
    metrics_dict[f"AUC_class_{c}"] = [roc_auc[c]]

df_metrics = pd.DataFrame(metrics_dict)
df_metrics.to_csv("test_metrics.csv", index=False)

# Confusion matrix CSV
df_cm = pd.DataFrame(
    cm,
    index=[f"True_{c}" for c in range(num_classes)],
    columns=[f"Pred_{c}" for c in range(num_classes)]
)
df_cm.to_csv("confusion_matrix.csv")

print("\nSaved: test_metrics.csv and confusion_matrix.csv")
print("ROC curve displayed.")


In [ ]:
torch.save(model.state_dict(), "glaucoma_model.pth")
print("Model saved as glaucoma_model.pth")